# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. We will walk through data loading, overview, extraction, exploratory analysis, and visualization steps referencing entities by their Croissant `@id` fields for reproducibility and clarity.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

We define the Croissant schema URL and load the metadata. Metadata provides a human readable overview of the dataset and its fields.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata as an object
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published on: {dataset.metadata.datePublished}\nVersion: {dataset.metadata.version}")
print(f"Croissant JSON-LD URL: {croissant_url}")

## 2. Data Overview
Review available record sets and their fields. We reference all entities using their Croissant `@id` for reproducibility.

Let's list available record sets and their fields.

In [ ]:
# Record set metadata overview
print("Available record sets and their fields (@id references):\n")
record_set_ids = []
for record_set in dataset.metadata.recordSets:
    rec_id = record_set.id
    record_set_ids.append(rec_id)
    print(f"RecordSet @id: {rec_id}, name: {record_set.name}")
    for field in record_set.fields:
        print(f"   Field @id: {field.id}, name: {field.name}, dataType: {field.dataType}")

To see some sample records, let's iterate through the first available record set.

In [ ]:
# Display a few records for the first record set
if len(record_set_ids) > 0:
    example_record_set_id = record_set_ids[0]
    print(f"\nFirst records (up to 2) from RecordSet @id: {example_record_set_id}\n---")
    for i, row in enumerate(dataset.records(record_set=example_record_set_id)):
        print(row)
        if i >= 1:
            break
else:
    print("No record sets defined in metadata.")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis.

We use the record set and field `@id`s discovered in the data overview step. All entities are referenced by their Croissant `@id`.

In [ ]:
# Extract data from all available record sets
dataframes = {}
for recset_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"Columns in DataFrame (field @id): {df.columns.tolist()}")
        display(df.head(2))
    else:
        print(f"No records found for {recset_id}.")

# For downstream sections, select the primary data record set
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
We perform typical analysis such as filtering, normalization, and grouping using column `@id` names for clarity and reproducibility.

First, let's list all numeric fields in the primary record set.

In [ ]:
# Identify likely numeric fields and group fields based on field metadata
numeric_field_ids = []
group_field_ids = []
if main_record_set_id is not None:
    for record_set in dataset.metadata.recordSets:
        if record_set.id == main_record_set_id:
            for field in record_set.fields:
                if field.dataType in ["schema:Float", "schema:Integer", "schema:Number"]:
                    numeric_field_ids.append(field.id)
                if field.dataType == "schema:Text":
                    group_field_ids.append(field.id)

    print(f"Numeric field options (@id): {numeric_field_ids}")
    print(f"Text/group field options (@id): {group_field_ids}")
else:
    print("No primary record set available.")

We'll select a numeric field for filtering and normalization, and a text field for grouping, both by `@id`.

You can change these variables to choose a different field as needed.

In [ ]:
if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id].copy()
    # Choose first detected numeric and group fields
    numeric_field_id = numeric_field_ids[0] if numeric_field_ids else df.columns[0]
    group_field_id = group_field_ids[0] if group_field_ids else None
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
    print(f"Filtering rows where {numeric_field_id} > {threshold:.3f}")
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field if present
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and the means by group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and main_record_set_id in dataframes:
    fig, axs = plt.subplots(1, 2, figsize=(12,4))
    ax1, ax2 = axs
    # Distribution plot
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], kde=True, ax=ax1)
    ax1.set_title(f"Distribution of {numeric_field_id}")
    # If grouping was performed
    if group_field_id is not None and 'grouped_df' in locals():
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ax=ax2)
        ax2.set_title(f"Mean {numeric_field_id} by {group_field_id}")
        ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
    else:
        ax2.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
We successfully loaded and explored the Mass Spectrometry EV Human Mast Cell dataset using the `mlcroissant` library.

- Dataset entities and operations referenced by Croissant `@id` ensure reproducibility.
- We surveyed the record sets and fields, loaded the main data into DataFrames, and performed basic filtering, normalization, and grouping.
- Visualizations illustrated core distributions and group statistics.

Explore further by iterating through additional fields, performing advanced EDA, or linking to other Croissant-compatible tools!